# Module 06: Adaptive Networks (Physarum) & Contagion / Flocking Dynamics
Reconstruction of **Tero et al. (2010, Science) Physarum Tokyo Rail Solver (Cost, Global Efficiency, Fault Tolerance, and Edge Disruption Recovery)**, plus comparative labs on **Simple vs. Complex Contagion (Centola, 2010)** and **Vicsek Collective Motion**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from scipy.sparse.csgraph import minimum_spanning_tree, shortest_path
import ipywidgets as widgets

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

# ------------------------------------------------------------------------------
# 1. TOKYO-INSPIRED PLANAR RAIL BENCHMARK & PHYSARUM HYDRAULIC SOLVER
# ------------------------------------------------------------------------------

def build_tokyo_benchmark_graph():
    '''Creates a 24-station planar urban rail benchmark centered on Tokyo Hub (node 0).'''
    coords = np.array([
        [0.52, 0.48],  # 0: Tokyo Central Hub
        [0.45, 0.52], [0.43, 0.45], [0.48, 0.40], [0.56, 0.42], [0.58, 0.54],
        [0.36, 0.58], [0.34, 0.47], [0.37, 0.36], [0.46, 0.30], [0.59, 0.31],
        [0.67, 0.43], [0.66, 0.58], [0.55, 0.65], [0.44, 0.66], [0.25, 0.68],
        [0.22, 0.50], [0.24, 0.32], [0.35, 0.20], [0.52, 0.18], [0.71, 0.24],
        [0.78, 0.46], [0.75, 0.68], [0.58, 0.78]
    ], dtype=np.float64)
    tri = Delaunay(coords)
    n_nodes = len(coords)
    adj_len = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    edges = set()
    for simplex in tri.simplices:
        for idx_a in range(3):
            u, v = int(simplex[idx_a]), int(simplex[(idx_a + 1) % 3])
            dist = float(np.linalg.norm(coords[u] - coords[v]))
            if dist < 0.36:
                edge = (min(u, v), max(u, v))
                edges.add(edge)
                adj_len[u, v] = adj_len[v, u] = dist
    return coords, sorted(list(edges)), adj_len

def solve_physarum_network(coords, edges, adj_len, gamma=1.4, steps=140, disrupt_hub_edges=False, seed=42):
    '''Solves Tero et al. (2010) adaptive tube conductivity dynamics on planar graph.'''
    rng = np.random.default_rng(seed)
    n_nodes = len(coords)
    n_edges = len(edges)
    D = np.ones(n_edges, dtype=np.float64) * 0.5
    lengths = np.array([adj_len[u, v] for (u, v) in edges], dtype=np.float64)
    disrupted_mask = np.zeros(n_edges, dtype=bool)

    for step in range(steps):
        # Optional experimental extension: sever top 2 high-flow trunk lines at midpoint
        if disrupt_hub_edges and step == steps // 2:
            top_two = np.argsort(D)[-2:]
            disrupted_mask[top_two] = True
            D[top_two] = 1e-6

        # Random source-sink pair (weighted toward central hub 0)
        src = 0 if rng.random() < 0.5 else int(rng.integers(0, n_nodes))
        snk = int(rng.integers(0, n_nodes))
        while snk == src:
            snk = int(rng.integers(0, n_nodes))

        # Build weighted Laplacian L_ij = -D_ij / L_ij
        L_mat = np.zeros((n_nodes, n_nodes), dtype=np.float64)
        conductance = D / lengths
        for e_idx, (u, v) in enumerate(edges):
            c = conductance[e_idx]
            L_mat[u, u] += c
            L_mat[v, v] += c
            L_mat[u, v] -= c
            L_mat[v, u] -= c

        # Ground sink node (p[snk] = 0) and solve reduced linear system
        rhs = np.zeros(n_nodes, dtype=np.float64)
        rhs[src] = 1.0
        keep = [i for i in range(n_nodes) if i != snk]
        L_red = L_mat[np.ix_(keep, keep)] + np.eye(len(keep)) * 1e-9
        p_red = np.linalg.solve(L_red, rhs[keep])
        pressures = np.zeros(n_nodes, dtype=np.float64)
        pressures[keep] = p_red

        # Compute fluxes Q_ij and update tube diameters D_ij
        flux = np.array([conductance[e_idx] * abs(pressures[u] - pressures[v]) for e_idx, (u, v) in enumerate(edges)])
        f_pow = flux ** gamma
        dt = 0.25
        D = D + dt * ((f_pow / (1.0 + f_pow)) - D)
        D[disrupted_mask] = 1e-6
        D = np.clip(D, 1e-6, 5.0)

    return D, disrupted_mask

def compute_network_metrics(n_nodes, edges, lengths, active_mask):
    '''Computes Total Length (Cost), Global Efficiency, and Single-Edge Fault Tolerance.'''
    dist_mat = np.zeros((n_nodes, n_nodes), dtype=np.float64)
    active_edges = []
    total_cost = 0.0
    for e_idx, (u, v) in enumerate(edges):
        if active_mask[e_idx]:
            dist_mat[u, v] = dist_mat[v, u] = lengths[e_idx]
            total_cost += lengths[e_idx]
            active_edges.append((u, v, lengths[e_idx]))

    sp = shortest_path(dist_mat, directed=False)
    inv_sp = np.where((sp > 0) & np.isfinite(sp), 1.0 / sp, 0.0)
    global_eff = float(np.sum(inv_sp) / (n_nodes * (n_nodes - 1)))

    # Fault tolerance: fraction of single-edge removals that keep graph connected
    if len(active_edges) == 0:
        return total_cost, global_eff, 0.0
    connected_survivals = 0
    for drop_idx in range(len(active_edges)):
        sub_mat = np.zeros((n_nodes, n_nodes), dtype=np.float64)
        for idx_e, (u, v, w) in enumerate(active_edges):
            if idx_e != drop_idx:
                sub_mat[u, v] = sub_mat[v, u] = w
        sub_sp = shortest_path(sub_mat, directed=False)
        if np.all(np.isfinite(sub_sp)):
            connected_survivals += 1
    fault_tol = connected_survivals / float(len(active_edges))
    return total_cost, global_eff, fault_tol

# ------------------------------------------------------------------------------
# 2. SIMPLE VS COMPLEX CONTAGION & VICSEK FLOCKING MODELS
# ------------------------------------------------------------------------------

def simulate_contagion_comparison(rewire_prob=0.15, complex_threshold=2, n_nodes=120, seed=42):
    '''Compares Simple (SIR-like) vs Complex (Threshold) Contagion on Watts-Strogatz graph.'''
    rng = np.random.default_rng(seed)
    adj = np.zeros((n_nodes, n_nodes), dtype=np.int32)
    # Ring lattice with k=4 neighbors
    for i in range(n_nodes):
        for offset in (1, 2):
            j = (i + offset) % n_nodes
            adj[i, j] = adj[j, i] = 1
    # Rewire edges with probability rewire_prob
    for i in range(n_nodes):
        for offset in (1, 2):
            if rng.random() < rewire_prob:
                j = (i + offset) % n_nodes
                adj[i, j] = adj[j, i] = 0
                new_j = int(rng.integers(0, n_nodes))
                if new_j != i:
                    adj[i, new_j] = adj[new_j, i] = 1

    steps = 35
    # Seed a localized cluster of 3 adjacent adopters
    simple_state = np.zeros(n_nodes, dtype=np.int32)
    complex_state = np.zeros(n_nodes, dtype=np.int32)
    simple_state[:3] = 1
    complex_state[:3] = 1
    simple_curve, complex_curve = [3 / float(n_nodes)], [3 / float(n_nodes)]

    for _ in range(steps):
        active_s = adj.dot(simple_state)
        new_s = (rng.random(n_nodes) < (1.0 - (1.0 - 0.18) ** active_s)).astype(np.int32)
        simple_state = np.maximum(simple_state, new_s)

        active_c = adj.dot(complex_state)
        new_c = (active_c >= complex_threshold).astype(np.int32)
        complex_state = np.maximum(complex_state, new_c)

        simple_curve.append(np.mean(simple_state))
        complex_curve.append(np.mean(complex_state))
    return np.array(simple_curve), np.array(complex_curve)

def simulate_vicsek_flocking(n_birds=140, noise_eta=0.6, steps=90, seed=42):
    '''Simulates 2D Vicsek self-propelled particles and computes polarization order parameter.'''
    rng = np.random.default_rng(seed)
    box_l = 10.0
    r_int = 1.3
    v0 = 0.25
    pos = rng.uniform(0, box_l, size=(n_birds, 2))
    angles = rng.uniform(-np.pi, np.pi, size=n_birds)
    order_hist = []

    for _ in range(steps):
        dx = pos[:, 0:1] - pos[:, 0:1].T
        dy = pos[:, 1:2] - pos[:, 1:2].T
        dx -= box_l * np.round(dx / box_l)
        dy -= box_l * np.round(dy / box_l)
        dist_sq = dx ** 2 + dy ** 2
        neighbors = dist_sq <= (r_int ** 2)
        mean_sin = neighbors.dot(np.sin(angles))
        mean_cos = neighbors.dot(np.cos(angles))
        angles = np.arctan2(mean_sin, mean_cos) + rng.uniform(-noise_eta / 2.0, noise_eta / 2.0, size=n_birds)
        pos[:, 0] = (pos[:, 0] + v0 * np.cos(angles)) % box_l
        pos[:, 1] = (pos[:, 1] + v0 * np.sin(angles)) % box_l
        phi = np.sqrt(np.mean(np.cos(angles)) ** 2 + np.mean(np.sin(angles)) ** 2)
        order_hist.append(phi)
    return pos, angles, np.array(order_hist)

# ------------------------------------------------------------------------------
# 3. INTERACTIVE MODULE 06 DASHBOARD
# ------------------------------------------------------------------------------

def run_module06_lab(mode='1. Physarum Tokyo Rail Solver', gamma_or_noise=1.4, disrupt_or_rewire=False):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.9))
    fig.patch.set_facecolor('#F8FAFC')

    if mode == '1. Physarum Tokyo Rail Solver':
        coords, edges, adj_len = build_tokyo_benchmark_graph()
        lengths = np.array([adj_len[u, v] for (u, v) in edges])
        D_final, disrupted = solve_physarum_network(
            coords, edges, adj_len, gamma=gamma_or_noise, steps=150, disrupt_hub_edges=disrupt_or_rewire, seed=42
        )
        active_phy = D_final > 0.12
        mst_sparse = minimum_spanning_tree(adj_len).toarray()
        active_mst = np.array([(mst_sparse[u, v] > 0 or mst_sparse[v, u] > 0) for (u, v) in edges])

        c_phy, e_phy, ft_phy = compute_network_metrics(len(coords), edges, lengths, active_phy)
        c_mst, e_mst, ft_mst = compute_network_metrics(len(coords), edges, lengths, active_mst)
        c_raw, e_raw, ft_raw = compute_network_metrics(len(coords), edges, lengths, np.ones(len(edges), dtype=bool))

        # Panel 1: Reconstructed Adaptive Network Topology
        axes[0].set_facecolor('#F8FAFC')
        for e_idx, (u, v) in enumerate(edges):
            if disrupted[e_idx]:
                axes[0].plot([coords[u, 0], coords[v, 0]], [coords[u, 1], coords[v, 1]], 'r--', linewidth=1.8, alpha=0.8)
            elif D_final[e_idx] > 0.12:
                lw = 0.8 + 3.2 * (D_final[e_idx] / np.max(D_final))
                axes[0].plot([coords[u, 0], coords[v, 0]], [coords[u, 1], coords[v, 1]], color='#0D9488', linewidth=lw)
            else:
                axes[0].plot([coords[u, 0], coords[v, 0]], [coords[u, 1], coords[v, 1]], color='#CBD5E1', linewidth=0.6, alpha=0.5)
        axes[0].scatter(coords[1:, 0], coords[1:, 1], color='#0F172A', s=35, zorder=5, label='Regional Stations')
        axes[0].scatter([coords[0, 0]], [coords[0, 1]], color='#E11D48', s=110, marker='*', zorder=6, label='Central Hub')
        axes[0].set_title(f'1. Physarum Network (gamma={gamma_or_noise:.2f} | Cut={disrupt_or_rewire})', fontsize=11, fontweight='bold')
        axes[0].legend(loc='lower left', facecolor='white')
        axes[0].set_xticks([])
        axes[0].set_yticks([])

        # Panel 2: Cost vs Efficiency vs Fault Tolerance Trade-off
        axes[1].set_facecolor('#F8FAFC')
        metrics_names = ['Normalized Cost', 'Global Efficiency', 'Fault Tolerance']
        phy_vals = [c_phy / c_mst, e_phy / e_mst, ft_phy]
        mst_vals = [1.0, 1.0, ft_mst]
        raw_vals = [c_raw / c_mst, e_raw / e_mst, ft_raw]
        x = np.arange(3)
        axes[1].bar(x - 0.22, mst_vals, width=0.22, color='#64748B', edgecolor='#0F172A', label='MST Baseline')
        axes[1].bar(x, phy_vals, width=0.22, color='#0D9488', edgecolor='#0F172A', label='Physarum Adaptive')
        axes[1].bar(x + 0.22, raw_vals, width=0.22, color='#D97706', edgecolor='#0F172A', label='Full Mesh')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(metrics_names, fontweight='bold')
        axes[1].set_title('2. Tero et al. (2010) Multi-Objective Benchmark', fontsize=11, fontweight='bold')
        axes[1].legend(facecolor='white')
        axes[1].grid(axis='y', linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print(f'Physarum Cost/MST = {c_phy/c_mst:.2f}x | Efficiency/MST = {e_phy/e_mst:.2f}x | Single-Edge Fault Tolerance = {ft_phy:.1%} (vs MST {ft_mst:.1%})')

    elif mode == '2. Simple vs. Complex Contagion':
        rewire_p = 0.35 if disrupt_or_rewire else 0.02
        s_curve, c_curve = simulate_contagion_comparison(rewire_prob=rewire_p, complex_threshold=2, n_nodes=120, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        axes[0].plot(s_curve, color='#2563EB', linewidth=2.4, label='Simple Contagion (SIR-like)')
        axes[0].plot(c_curve, color='#E11D48', linewidth=2.4, linestyle='--', label='Complex Contagion (Threshold >= 2)')
        axes[0].set_ylim(0, 1.05)
        axes[0].set_title(f'1. Contagion Adoption Curves (Rewiring p = {rewire_p:.2f})', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('Time Step t')
        axes[0].set_ylabel('Fraction of Adopters')
        axes[0].legend(facecolor='white')
        axes[0].grid(True, linestyle='--', alpha=0.4)

        # Compare final adoption across rewiring probabilities (Centola, 2010)
        p_sweep = [0.0, 0.05, 0.15, 0.30, 0.50]
        s_final, c_final = [], []
        for p_val in p_sweep:
            sc, cc = simulate_contagion_comparison(rewire_prob=p_val, complex_threshold=2, n_nodes=120, seed=42)
            s_final.append(sc[-1])
            c_final.append(cc[-1])
        axes[1].set_facecolor('#F8FAFC')
        axes[1].plot(p_sweep, s_final, marker='o', color='#2563EB', linewidth=2.2, label='Simple Contagion Reach')
        axes[1].plot(p_sweep, c_final, marker='s', color='#E11D48', linewidth=2.2, label='Complex Contagion Reach')
        axes[1].set_ylim(0, 1.05)
        axes[1].set_title('2. Effect of Small-World Shortcuts (Centola, 2010)', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Network Rewiring Probability p')
        axes[1].set_ylabel('Final Reach at t = 35')
        axes[1].legend(facecolor='white')
        axes[1].grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print('INSIGHT: Random shortcuts accelerate Simple Contagion, but destroy local clustering needed for Complex Contagion!')

    else:
        # Mode 3: Vicsek Collective Flocking
        noise_val = min(max(gamma_or_noise * 2.0, 0.1), 5.5)
        pos, angles, order_hist = simulate_vicsek_flocking(n_birds=140, noise_eta=noise_val, steps=90, seed=42)
        axes[0].set_facecolor('#F8FAFC')
        axes[0].quiver(pos[:, 0], pos[:, 1], np.cos(angles), np.sin(angles), color='#2563EB', scale=28)
        axes[0].set_xlim(0, 10)
        axes[0].set_ylim(0, 10)
        axes[0].set_title(f'1. Vicsek Self-Propelled Particles (Noise eta = {noise_val:.2f})', fontsize=11, fontweight='bold')
        axes[0].set_xlabel('X Position')
        axes[0].set_ylabel('Y Position')

        axes[1].set_facecolor('#F8FAFC')
        axes[1].plot(order_hist, color='#0D9488', linewidth=2.2)
        axes[1].set_ylim(0, 1.05)
        axes[1].set_title(f'2. Polarization Order Parameter Phi = {order_hist[-1]:.3f}', fontsize=11, fontweight='bold')
        axes[1].set_xlabel('Time Step t')
        axes[1].set_ylabel('Order Parameter Phi')
        axes[1].grid(True, linestyle='--', alpha=0.4)
        plt.tight_layout()
        plt.show()
        print(f'Vicsek Flocking | Noise eta = {noise_val:.2f} | Final Polarization Order Parameter Phi = {order_hist[-1]:.3f}')

widgets.interact(
    run_module06_lab,
    mode=widgets.Dropdown(options=['1. Physarum Tokyo Rail Solver', '2. Simple vs. Complex Contagion', '3. Vicsek Collective Flocking'], value='1. Physarum Tokyo Rail Solver', description='Lab Study:'),
    gamma_or_noise=widgets.FloatSlider(min=0.5, max=2.2, step=0.1, value=1.4, description='Gamma / Noise:'),
    disrupt_or_rewire=widgets.Checkbox(value=False, description='Cut Hub Edges / High Rewiring'),
);
